# 04-2. LLM 기반 출시 후 패치·운영 전략 제안

04번에서 Steam 리뷰를 LLM으로 분류하고, 04-1번에서 분류 결과를 전처리했다.  
이 노트북은 04-1에서 만든 근거 데이터를 바탕으로 **Tainted Grail: The Fall of Avalon의 출시 후 패치·운영 전략 초안**을 생성한다.

## 분석 흐름

```text
04. 출시 후 리뷰 LLM 분류
→ 04-1. 분류된 리뷰 전처리
→ 04-2. LLM 기반 패치·운영 전략 제안
```

## 핵심 원칙

- LLM은 최종 우선순위를 새로 판단하지 않는다.
- 04-1에서 계산한 `rule_priority_hint`, `action_group_hint`를 고정 근거로 사용한다.
- LLM은 이미 집계된 근거를 바탕으로 패치·운영 문장을 작성하는 역할만 한다.
- `High urgency`는 리뷰 내용을 보고 LLM이 분류한 시급도 후보이므로, 최종 우선순위의 단독 기준으로 사용하지 않는다.

# 0. 환경설정

In [1]:

# ============================================================
# 기본 라이브러리
# ============================================================
import os
import json
import platform
from pathlib import Path
from datetime import datetime
from typing import List, Literal

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# LLM / Pydantic 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 200)

# 1. 기본 설정

In [2]:

# ============================================================
# 프로젝트 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 04_run_llm_postlaunch_tainted-grail_analysis.ipynb에서 사용한 실행 이름과 맞춘다.
# ============================================================
RUN_NAME = "postlaunch_tainted-grail"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME

# ============================================================
# 04-1 전처리 산출물 폴더
# ============================================================
POSTLAUNCH_PREPROCESS_DIR = OUTPUT_DIR / "postlaunch_preprocess_data"

REVIEW_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_review_base.csv"
ISSUE_SUMMARY_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_issue_summary.csv"
PATCH_OPS_EVIDENCE_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"
TABLEAU_POSTLAUNCH_SOURCE_PATH = POSTLAUNCH_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"

# ============================================================
# 이번 04-2 노트북에서 생성할 산출물 저장 폴더
# ============================================================
PATCH_OPS_STRATEGY_DIR = OUTPUT_DIR / "postlaunch_patch_ops_strategy_data"
PATCH_OPS_STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

PATCH_OPS_STRATEGY_PATH = PATCH_OPS_STRATEGY_DIR / "postlaunch_patch_ops_strategy.csv"
PATCH_OPS_RESULT_JSON_PATH = PATCH_OPS_STRATEGY_DIR / "postlaunch_patch_ops_strategy_result.json"
PATCH_OPS_REPORT_TEXT_PATH = PATCH_OPS_STRATEGY_DIR / "postlaunch_patch_ops_report_text.md"
PATCH_OPS_PROMPT_PATH = PATCH_OPS_STRATEGY_DIR / "postlaunch_patch_ops_prompt.txt"

print("ROOT:", ROOT)
print("RUN_NAME:", RUN_NAME)
print("04-1 전처리 폴더:", POSTLAUNCH_PREPROCESS_DIR)
print("04-2 저장 폴더:", PATCH_OPS_STRATEGY_DIR)

ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
RUN_NAME: postlaunch_tainted-grail
04-1 전처리 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\postlaunch_preprocess_data
04-2 저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\postlaunch_patch_ops_strategy_data


In [3]:

# ============================================================
# LLM 실행 여부
# ============================================================
# False: LLM 호출 없이 근거 데이터와 프롬프트만 확인한다.
# True : 실제 LLM을 호출해서 패치·운영 전략 초안을 생성한다.

RUN_PATCH_OPS_LLM = True

# ============================================================
# 분석 대상 게임
# ============================================================
TARGET_APPID = 1466060
TARGET_GAME_NAME = "Tainted Grail: The Fall of Avalon"

# ============================================================
# 프롬프트에 넣을 근거 데이터 개수
# ============================================================
# 04-1에서 정리한 이슈 근거 중 상위 이슈를 LLM에게 제공한다.
# 전체 이슈를 모두 넣으면 프롬프트가 길어질 수 있으므로, 대응 구분별로 대표 이슈를 우선 사용한다.

MAX_ISSUES_FOR_PROMPT = 18
MAX_EVIDENCE_TEXT_ISSUES = 12

GROUP_LIMITS = {
    "즉시 확인": 5,
    "단기 개선": 6,
    "장기 검토": 4,
    "강점 유지": 3,
    "검토 필요": 2,
}

# ============================================================
# LLM 호출 설정
# ============================================================
MAX_RETRIES = 3
TEMPERATURE = 0.0

print("RUN_PATCH_OPS_LLM:", RUN_PATCH_OPS_LLM)
print("분석 대상 게임:", TARGET_GAME_NAME)
print("TARGET_APPID:", TARGET_APPID)
print("MAX_ISSUES_FOR_PROMPT:", MAX_ISSUES_FOR_PROMPT)

RUN_PATCH_OPS_LLM: True
분석 대상 게임: Tainted Grail: The Fall of Avalon
TARGET_APPID: 1466060
MAX_ISSUES_FOR_PROMPT: 18


# 2. Vertex AI / PydanticAI 설정

In [4]:

# ============================================================
# Vertex AI 설정
# ============================================================
# 04_run_llm_postlaunch_tainted-grail_analysis.ipynb와 같은 방식으로 구성한다.

load_dotenv()
load_dotenv(ROOT / ".env")

GOOGLE_CLOUD_PROJECT = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GCP_PROJECT_ID")
)

GOOGLE_CLOUD_LOCATION = (
    os.getenv("GOOGLE_CLOUD_LOCATION")
    or os.getenv("VERTEX_LOCATION")
    or "global"
)

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")

if GOOGLE_CLOUD_PROJECT:
    vertex_provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )

    vertex_model = GoogleModel(
        GEMINI_MODEL,
        provider=vertex_provider,
    )

    print("Vertex AI project:", GOOGLE_CLOUD_PROJECT)
    print("Vertex AI location:", GOOGLE_CLOUD_LOCATION)
    print("Gemini model:", GEMINI_MODEL)
    print("Vertex 모델 생성: O")
else:
    vertex_provider = None
    vertex_model = None

    print("Vertex AI project: 미설정")
    print("Vertex 모델 생성: X")
    print("실제 LLM 실행 전 .env에 GOOGLE_CLOUD_PROJECT를 설정하세요.")

Vertex AI project: gen-lang-client-0587784564
Vertex AI location: global
Gemini model: gemini-3.1-flash-lite-preview
Vertex 모델 생성: O


# 3. 공통 함수

In [5]:

# ============================================================
# JSON 변환 보조 함수
# ============================================================

def to_serializable(obj):
    """Pydantic, pandas, numpy 값을 JSON 저장 가능한 기본 타입으로 바꾼다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def save_json_safely(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_serializable(obj), f, ensure_ascii=False, indent=2)
    print("JSON 저장:", path)


# ============================================================
# 프롬프트용 텍스트 표 생성 함수
# ============================================================

def shorten_text(value, max_chars=180):
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\n", " ").strip()
    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."
    return text


def df_to_text_table(df, columns, max_rows=20, max_cell_chars=180):
    """tabulate 없이 LLM 프롬프트에 넣을 간단 텍스트 표를 만든다."""
    if df is None or len(df) == 0:
        return "해당 조건에 맞는 근거 데이터가 없습니다."

    cols = [col for col in columns if col in df.columns]
    small = df[cols].head(max_rows).copy()

    lines = []
    header = " | ".join(cols)
    lines.append(header)
    lines.append("-" * len(header))

    for _, row in small.iterrows():
        values = [shorten_text(row.get(col, ""), max_cell_chars) for col in cols]
        lines.append(" | ".join(values))

    return "\n".join(lines)


# ============================================================
# 정렬 보조 함수
# ============================================================

def add_order_columns(df):
    out = df.copy()
    priority_order_map = {"상": 1, "중": 2, "하": 3}
    action_order_map = {
        "즉시 확인": 1,
        "단기 개선": 2,
        "장기 검토": 3,
        "강점 유지": 4,
        "검토 필요": 5,
    }

    out["priority_order"] = out.get("rule_priority_hint", "").map(priority_order_map).fillna(9)
    out["action_order"] = out.get("action_group_hint", "").map(action_order_map).fillna(9)
    return out

# 4. 데이터 불러오기

In [6]:

# ============================================================
# 04-1에서 만든 CSV 불러오기
# ============================================================

review_base = pd.read_csv(REVIEW_BASE_PATH)
issue_summary = pd.read_csv(ISSUE_SUMMARY_PATH)
evidence_base = pd.read_csv(PATCH_OPS_EVIDENCE_BASE_PATH)
tableau_source = pd.read_csv(TABLEAU_POSTLAUNCH_SOURCE_PATH)

# 문자열 컬럼 정리
for df in [review_base, issue_summary, evidence_base, tableau_source]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# 날짜 컬럼 변환
for col in ["review_datetime", "release_date"]:
    if col in review_base.columns:
        review_base[col] = pd.to_datetime(review_base[col], errors="coerce")
    if col in tableau_source.columns:
        tableau_source[col] = pd.to_datetime(tableau_source[col], errors="coerce")

print("review_base:", review_base.shape)
print("issue_summary:", issue_summary.shape)
print("evidence_base:", evidence_base.shape)
print("tableau_source:", tableau_source.shape)

display(evidence_base.head())

review_base: (1000, 28)
issue_summary: (21, 24)
evidence_base: (21, 25)
tableau_source: (1967, 32)


C:\Users\joon5\AppData\Local\Temp\ipykernel_14584\3146103069.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\joon5\AppData\Local\Temp\ipykernel_14584\3146103069.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

,llm_issue_category,appid,game_name,issue_name_kor,affected_review_count,positive_review_count,negative_review_count,mixed_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,high_urgency_negative_mixed_review_count,recent_30d_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,avg_playtime_at_review_hours,median_playtime_at_review_hours,high_urgency_rate,negative_mixed_rate,recent_30d_negative_mixed_rate,action_group_hint,rule_priority_hint,priority_reason,patch_ops_note,llm_evidence_text
0,bug,1466060,Tainted Grail: The Fall of Avalon,버그,95,4,85,0,85,41,38,38,38,33,11,44.86,34.28,0.4000,0.8947,0.8684,즉시 확인,상,"영향 리뷰 95개, 부정·혼합 85개, High urgency 38개, 최근 30일 부정·혼합 33개, 초반 플레이타임 부정·혼합 11개","반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.",[ISSUE]\nissue_category: bug\nissue_name_kor: 버그\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\naffected_review_count: 95\nnegative_mixed_review_count: 85\nhigh_urgency_review_count: 38\nhigh_u...
1,performance,1466060,Tainted Grail: The Fall of Avalon,성능,60,5,53,0,53,26,26,26,25,22,13,26.13,19.30,0.4333,0.8833,0.8800,즉시 확인,상,"영향 리뷰 60개, 부정·혼합 53개, High urgency 26개, 최근 30일 부정·혼합 22개, 초반 플레이타임 부정·혼합 13개","프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.",[ISSUE]\nissue_category: performance\nissue_name_kor: 성능\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\naffected_review_count: 60\nnegative_mixed_review_count: 53\nhigh_urgency_review_count: 26...
2,crash,1466060,Tainted Grail: The Fall of Avalon,크래시,30,0,29,0,29,25,26,26,12,11,7,28.43,20.27,0.8667,0.9667,0.9167,즉시 확인,상,"영향 리뷰 30개, 부정·혼합 29개, High urgency 26개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 7개","크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.",[ISSUE]\nissue_category: crash\nissue_name_kor: 크래시\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\naffected_review_count: 30\nnegative_mixed_review_count: 29\nhigh_urgency_review_count: 26\nhig...
3,progression_grind,1466060,Tainted Grail: The Fall of Avalon,성장/반복 노가다,28,1,26,0,26,16,14,14,9,9,2,57.39,31.98,0.5000,0.9286,1.0000,즉시 확인,상,"영향 리뷰 28개, 부정·혼합 26개, High urgency 14개, 최근 30일 부정·혼합 9개, 초반 플레이타임 부정·혼합 2개",반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.,[ISSUE]\nissue_category: progression_grind\nissue_name_kor: 성장/반복 노가다\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\naffected_review_count: 28\nnegative_mixed_review_count: 26\nhigh_urgency_rev...
4,optimization,1466060,Tainted Grail: The Fall of Avalon,최적화,27,1,25,1,26,16,14,14,11,11,11,19.36,8.40,0.5185,0.9630,1.0000,즉시 확인,상,"영향 리뷰 27개, 부정·혼합 26개, High urgency 14개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 11개",최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.,[ISSUE]\nissue_category: optimization\nissue_name_kor: 최적화\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\naffected_review_count: 27\nnegative_mixed_review_count: 26\nhigh_urgency_review_count: ...


In [7]:

# ============================================================
# 필수 컬럼 확인
# ============================================================

required_evidence_cols = [
    "issue_name_kor",
    "action_group_hint",
    "rule_priority_hint",
    "affected_review_count",
    "negative_mixed_review_count",
    "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count",
    "early_playtime_negative_mixed_review_count",
    "priority_reason",
    "patch_ops_note",
    "llm_evidence_text",
]

missing_cols = [col for col in required_evidence_cols if col not in evidence_base.columns]
if missing_cols:
    raise ValueError(f"evidence_base에 필요한 컬럼이 없습니다: {missing_cols}")

print("필수 컬럼 확인 완료")

필수 컬럼 확인 완료


# 5. 분석 대상 요약 및 근거 데이터 선택

In [8]:

# ============================================================
# 분석 대상 기본 요약
# ============================================================

def value_count_text(series):
    counts = series.value_counts(dropna=False)
    return ", ".join([f"{idx}: {cnt}" for idx, cnt in counts.items()])

analysis_overview = {
    "game_name": TARGET_GAME_NAME,
    "appid": TARGET_APPID,
    "review_count": int(len(review_base)),
    "issue_tag_count": int(len(tableau_source)),
    "review_date_min": review_base["review_datetime"].min().strftime("%Y-%m-%d") if "review_datetime" in review_base.columns else "",
    "review_date_max": review_base["review_datetime"].max().strftime("%Y-%m-%d") if "review_datetime" in review_base.columns else "",
    "steam_label_distribution": value_count_text(review_base["steam_label_text"]) if "steam_label_text" in review_base.columns else "",
    "llm_sentiment_distribution": value_count_text(review_base["llm_sentiment"]) if "llm_sentiment" in review_base.columns else "",
    "high_urgency_review_count": int(review_base.get("high_urgency_flag", pd.Series(dtype=bool)).sum()) if "high_urgency_flag" in review_base.columns else 0,
    "recent_30d_review_count": int(review_base.get("recent_30d_flag", pd.Series(dtype=bool)).sum()) if "recent_30d_flag" in review_base.columns else 0,
}

print(json.dumps(analysis_overview, ensure_ascii=False, indent=2))

{
  "game_name": "Tainted Grail: The Fall of Avalon",
  "appid": 1466060,
  "review_count": 1000,
  "issue_tag_count": 1967,
  "review_date_min": "2026-01-29",
  "review_date_max": "2026-04-29",
  "steam_label_distribution": "positive: 671, negative: 329",
  "llm_sentiment_distribution": "positive: 592, negative: 302, mixed: 103, neutral: 3",
  "high_urgency_review_count": 178,
  "recent_30d_review_count": 373
}


In [9]:

# ============================================================
# 04-2 LLM 입력용 근거 데이터 선택
# ============================================================
# 목적:
# - LLM에게 모든 원천 데이터를 주는 것이 아니라, 04-1에서 집계된 이슈별 근거를 준다.
# - 대응 구분별 대표 이슈를 골라 프롬프트가 너무 길어지는 것을 막는다.
# - 최종 우선순위는 여기서 새로 판단하지 않고, 04-1의 rule_priority_hint를 그대로 사용한다.

work = add_order_columns(evidence_base)

sort_cols = [
    "action_order",
    "priority_order",
    "negative_mixed_review_count",
    "high_urgency_review_count",
    "affected_review_count",
]

work = work.sort_values(sort_cols, ascending=[True, True, False, False, False])

selected_parts = []
for group_name, limit in GROUP_LIMITS.items():
    part = work[work["action_group_hint"] == group_name].head(limit)
    selected_parts.append(part)

selected_evidence = (
    pd.concat(selected_parts, ignore_index=True)
    .drop_duplicates("issue_name_kor")
    .head(MAX_ISSUES_FOR_PROMPT)
    .copy()
)

selected_evidence = add_order_columns(selected_evidence)
selected_evidence = selected_evidence.sort_values(sort_cols, ascending=[True, True, False, False, False])

print("선택된 이슈 근거 수:", len(selected_evidence))
display(selected_evidence[[
    "issue_name_kor",
    "action_group_hint",
    "rule_priority_hint",
    "affected_review_count",
    "negative_mixed_review_count",
    "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count",
    "early_playtime_negative_mixed_review_count",
    "priority_reason",
]].head(20))

선택된 이슈 근거 수: 14


,issue_name_kor,action_group_hint,rule_priority_hint,affected_review_count,negative_mixed_review_count,high_urgency_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,priority_reason
0,버그,즉시 확인,상,95,85,38,33,11,"영향 리뷰 95개, 부정·혼합 85개, High urgency 38개, 최근 30일 부정·혼합 33개, 초반 플레이타임 부정·혼합 11개"
1,성능,즉시 확인,상,60,53,26,22,13,"영향 리뷰 60개, 부정·혼합 53개, High urgency 26개, 최근 30일 부정·혼합 22개, 초반 플레이타임 부정·혼합 13개"
2,크래시,즉시 확인,상,30,29,26,11,7,"영향 리뷰 30개, 부정·혼합 29개, High urgency 26개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 7개"
3,성장/반복 노가다,즉시 확인,상,28,26,14,9,2,"영향 리뷰 28개, 부정·혼합 26개, High urgency 14개, 최근 30일 부정·혼합 9개, 초반 플레이타임 부정·혼합 2개"
4,최적화,즉시 확인,상,27,26,14,11,11,"영향 리뷰 27개, 부정·혼합 26개, High urgency 14개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 11개"
5,게임플레이 루프,단기 개선,상,284,208,73,81,41,"영향 리뷰 284개, 부정·혼합 208개, High urgency 73개, 최근 30일 부정·혼합 81개, 초반 플레이타임 부정·혼합 41개"
6,밸런스,단기 개선,상,130,118,40,50,6,"영향 리뷰 130개, 부정·혼합 118개, High urgency 40개, 최근 30일 부정·혼합 50개, 초반 플레이타임 부정·혼합 6개"
7,UI/UX,단기 개선,상,63,58,20,19,8,"영향 리뷰 63개, 부정·혼합 58개, High urgency 20개, 최근 30일 부정·혼합 19개, 초반 플레이타임 부정·혼합 8개"
8,난이도,단기 개선,상,68,48,19,21,7,"영향 리뷰 68개, 부정·혼합 48개, High urgency 19개, 최근 30일 부정·혼합 21개, 초반 플레이타임 부정·혼합 7개"
9,그래픽/사운드,장기 검토,상,133,95,28,38,19,"영향 리뷰 133개, 부정·혼합 95개, High urgency 28개, 최근 30일 부정·혼합 38개, 초반 플레이타임 부정·혼합 19개"


# 6. LLM 출력 스키마 정의

In [10]:

# ============================================================
# 패치·운영 전략 출력 스키마
# ============================================================
# priority와 action_group은 LLM이 새로 정하는 값이 아니다.
# 04-1에서 만든 rule_priority_hint, action_group_hint를 그대로 복사하는 값이다.

class PatchOpsItem(BaseModel):
    action_group: Literal["즉시 확인", "단기 개선", "장기 검토", "강점 유지", "검토 필요"] = Field(
        description="근거표의 action_group_hint를 그대로 복사한 대응 구분"
    )
    priority: Literal["상", "중", "하"] = Field(
        description="근거표의 rule_priority_hint를 그대로 복사한 고정 우선순위"
    )
    issue_name: str = Field(
        description="근거가 된 이슈명. 반드시 근거표의 issue_name_kor 값 중 하나를 그대로 사용"
    )
    patch_ops_direction: str = Field(
        description="해당 이슈에 대한 패치·운영 방향 요약"
    )
    detailed_actions: List[str] = Field(
        description="실제로 검토할 세부 실행안 2~4개"
    )
    evidence_summary: str = Field(
        description="제공된 근거표의 수치와 대표 리뷰 근거를 바탕으로 한 요약"
    )
    expected_effect: str = Field(
        description="이 대응이 기대하는 유저 경험 개선 효과"
    )
    caution: str = Field(
        description="해석이나 실행 시 주의해야 할 점"
    )


class PostLaunchPatchOpsResult(BaseModel):
    title: str = Field(description="보고서 제목")
    game_summary: str = Field(description="분석 대상 게임 요약")
    data_summary: str = Field(description="분석 데이터 규모와 기간 요약")
    current_status_summary: str = Field(description="최근 리뷰 기준 현재 반응 상태 요약")
    immediate_actions: List[PatchOpsItem] = Field(description="action_group_hint가 즉시 확인인 항목")
    short_term_improvements: List[PatchOpsItem] = Field(description="action_group_hint가 단기 개선인 항목")
    long_term_reviews: List[PatchOpsItem] = Field(description="action_group_hint가 장기 검토 또는 검토 필요인 항목")
    strengths_to_keep: List[PatchOpsItem] = Field(description="action_group_hint가 강점 유지인 항목")
    operation_notes: List[str] = Field(description="패치 노트, 커뮤니티 공지, 모니터링 등 운영 관점 제안")
    cautions: List[str] = Field(description="해석 시 주의사항")
    final_summary: str = Field(description="전체 패치·운영 방향 요약")


print("패치·운영 전략 출력 스키마 정의 완료")

패치·운영 전략 출력 스키마 정의 완료


# 7. LLM 프롬프트 생성

In [11]:

# ============================================================
# 프롬프트 생성 함수
# ============================================================

def build_patch_ops_prompt(analysis_overview, selected_evidence_df):
    selected_evidence_df = selected_evidence_df.copy()

    evidence_cols = [
        "issue_name_kor",
        "action_group_hint",
        "rule_priority_hint",
        "affected_review_count",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "high_urgency_review_count",
        "recent_30d_negative_mixed_review_count",
        "early_playtime_negative_mixed_review_count",
        "high_urgency_rate",
        "negative_mixed_rate",
        "priority_reason",
        "patch_ops_note",
    ]
    evidence_cols = [col for col in evidence_cols if col in selected_evidence_df.columns]

    evidence_text = df_to_text_table(
        selected_evidence_df,
        columns=evidence_cols,
        max_rows=MAX_ISSUES_FOR_PROMPT,
        max_cell_chars=180,
    )

    review_evidence_text = "\n\n".join(
        selected_evidence_df["llm_evidence_text"].head(MAX_EVIDENCE_TEXT_ISSUES).tolist()
    )

    prompt = f"""
당신은 Steam 인디게임의 출시 후 패치·운영 전략을 정리하는 데이터 분석 보조자입니다.

목표:
Tainted Grail: The Fall of Avalon의 최근 Steam 리뷰를 LLM으로 분류하고,
04-1에서 집계한 이슈별 반복성, 부정·혼합 분포, High urgency 분포, 최근성, 플레이타임 근거를 바탕으로
패치·운영 전략 초안을 작성하세요.

가장 중요한 제한:
- 당신은 우선순위를 새로 계산하거나 판단하지 않습니다.
- 근거표의 action_group_hint는 이미 데이터 기준으로 계산된 대응 구분입니다.
- 근거표의 rule_priority_hint는 이미 데이터 기준으로 계산된 고정 우선순위입니다.
- 각 항목의 action_group은 반드시 근거표의 action_group_hint를 그대로 사용하세요.
- 각 항목의 priority는 반드시 근거표의 rule_priority_hint를 그대로 사용하세요.
- action_group_hint를 바꾸거나, rule_priority_hint를 올리거나 내리지 마세요.
- 근거표에 없는 issue_name_kor를 새로 만들지 마세요.
- issue_name에는 반드시 근거표에 있는 issue_name_kor 값을 그대로 작성하세요.

근거 사용 기준:
- affected_review_count는 해당 이슈가 언급된 리뷰 수입니다.
- negative_mixed_review_count는 LLM이 부정 또는 혼합 맥락으로 분류한 리뷰 수입니다.
- steam_negative_review_count는 Steam 비추천 리뷰 수입니다.
- high_urgency_review_count는 04번 리뷰 분류 단계에서 LLM이 High urgency 후보로 분류한 리뷰 수입니다.
- High urgency는 보조 지표이며, 단독으로 우선순위를 새로 정하는 기준이 아닙니다.
- recent_30d_negative_mixed_review_count는 최근 30일에도 부정·혼합 이슈가 반복되는지 보는 기준입니다.
- early_playtime_negative_mixed_review_count는 짧은 플레이타임에서 부정·혼합 이슈가 나타나는지 보는 기준입니다.
- 수치가 작거나 근거가 약한 경우에는 단정하지 말고 "검토 필요" 또는 "추가 확인 필요"라고 표현하세요.

작성 기준:
- 패치·운영 제안은 개발자가 실제로 실행할 수 있는 문장으로 작성하세요.
- 즉시 확인 항목은 재현, 로그 확인, 진행 차단 여부, 크래시/성능/저장 문제 확인처럼 구체적으로 작성하세요.
- 단기 개선 항목은 경험 품질을 낮추는 반복 문제를 다음 패치에서 점검하는 방향으로 작성하세요.
- 장기 검토 항목은 개발 범위가 큰 콘텐츠, 스토리, 구조 개선을 로드맵 관점으로 작성하세요.
- 강점 유지 항목은 업데이트와 커뮤니티 메시지에서 계속 살릴 요소로 작성하세요.
- 리뷰 수나 LLM 분류만으로 실제 버그 원인을 확정하지 마세요.
- "반드시 개선된다", "성공한다" 같은 보장 표현은 쓰지 마세요.

분석 대상 요약:
{json.dumps(analysis_overview, ensure_ascii=False, indent=2)}

이슈별 집계 근거표:
{evidence_text}

대표 리뷰 근거와 LLM 개선 제안 후보:
{review_evidence_text}

출력 요구:
1. 게임과 데이터 규모를 간단히 요약하세요.
2. 현재 리뷰 상태를 2~3문장으로 요약하세요.
3. immediate_actions에는 action_group_hint가 "즉시 확인"인 항목만 작성하세요.
4. short_term_improvements에는 action_group_hint가 "단기 개선"인 항목만 작성하세요.
5. long_term_reviews에는 action_group_hint가 "장기 검토" 또는 "검토 필요"인 항목만 작성하세요.
6. strengths_to_keep에는 action_group_hint가 "강점 유지"인 항목만 작성하세요.
7. 각 항목은 issue_name, priority, action_group, patch_ops_direction, detailed_actions, evidence_summary, expected_effect, caution을 포함해야 합니다.
8. evidence_summary에는 가능한 한 affected_review_count, negative_mixed_review_count, high_urgency_review_count, recent_30d_negative_mixed_review_count 중 2개 이상을 포함하세요.
9. operation_notes에는 패치 노트, 커뮤니티 공지, 모니터링 관점의 운영 제안을 작성하세요.
10. 마지막에는 해석 시 주의사항과 최종 요약을 작성하세요.
"""

    return prompt.strip()


patch_ops_prompt = build_patch_ops_prompt(
    analysis_overview=analysis_overview,
    selected_evidence_df=selected_evidence,
)

PATCH_OPS_PROMPT_PATH.write_text(patch_ops_prompt, encoding="utf-8")

print("프롬프트 생성 완료")
print("프롬프트 저장:", PATCH_OPS_PROMPT_PATH)
print(patch_ops_prompt[:2500])

프롬프트 생성 완료
프롬프트 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\postlaunch_patch_ops_strategy_data\postlaunch_patch_ops_prompt.txt
당신은 Steam 인디게임의 출시 후 패치·운영 전략을 정리하는 데이터 분석 보조자입니다.

목표:
Tainted Grail: The Fall of Avalon의 최근 Steam 리뷰를 LLM으로 분류하고,
04-1에서 집계한 이슈별 반복성, 부정·혼합 분포, High urgency 분포, 최근성, 플레이타임 근거를 바탕으로
패치·운영 전략 초안을 작성하세요.

가장 중요한 제한:
- 당신은 우선순위를 새로 계산하거나 판단하지 않습니다.
- 근거표의 action_group_hint는 이미 데이터 기준으로 계산된 대응 구분입니다.
- 근거표의 rule_priority_hint는 이미 데이터 기준으로 계산된 고정 우선순위입니다.
- 각 항목의 action_group은 반드시 근거표의 action_group_hint를 그대로 사용하세요.
- 각 항목의 priority는 반드시 근거표의 rule_priority_hint를 그대로 사용하세요.
- action_group_hint를 바꾸거나, rule_priority_hint를 올리거나 내리지 마세요.
- 근거표에 없는 issue_name_kor를 새로 만들지 마세요.
- issue_name에는 반드시 근거표에 있는 issue_name_kor 값을 그대로 작성하세요.

근거 사용 기준:
- affected_review_count는 해당 이슈가 언급된 리뷰 수입니다.
- negative_mixed_review_count는 LLM이 부정 또는 혼합 맥락으로 분류한 리뷰 수입니다.
- steam_negative_review_count는 Steam 비추천 리뷰 수입니다.
- high_urgency_revie

# 8. PydanticAI Agent 설정

In [12]:

# ============================================================
# 패치·운영 전략 생성 Agent
# ============================================================

system_prompt = """
당신은 Steam 인디게임의 출시 후 패치·운영 전략을 정리하는 데이터 분석 보조자입니다.

당신의 역할은 우선순위 판단이 아니라 문장화와 전략 초안 작성입니다.
제공된 근거표의 action_group_hint와 rule_priority_hint를 반드시 그대로 사용하세요.
action_group_hint를 바꾸거나, rule_priority_hint를 새로 계산하거나, 올리거나, 내리지 마세요.
근거표에 없는 issue_name_kor를 새로 만들지 마세요.
issue_name에는 근거표의 issue_name_kor 값을 그대로 작성하세요.
High urgency는 이전 LLM 리뷰 분류 결과를 집계한 보조 지표이므로, 단독 우선순위 기준으로 사용하지 마세요.
리뷰 수, 부정·혼합 리뷰 수, 최근 30일 반복 여부, 짧은 플레이타임 이슈를 함께 언급하세요.
실제 원인이 확정된 것처럼 단정하지 말고, 패치와 운영에서 확인해야 할 방향으로 표현하세요.
"""

patch_ops_settings = GoogleModelSettings(
    temperature=TEMPERATURE,
)

if vertex_model is not None:
    patch_ops_agent = Agent(
        vertex_model,
        output_type=PostLaunchPatchOpsResult,
        system_prompt=system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    patch_ops_agent = None

print("PydanticAI Agent 생성 여부:", "O" if patch_ops_agent is not None else "X")

PydanticAI Agent 생성 여부: O


# 9. LLM 호출 전 확인

In [13]:

# ============================================================
# LLM 호출 전 확인용 요약
# ============================================================
# RUN_PATCH_OPS_LLM=False 상태에서도 근거 데이터와 프롬프트가 잘 잡혔는지 확인한다.

print("분석 대상:", analysis_overview["game_name"])
print("분석 리뷰 수:", analysis_overview["review_count"])
print("분석 리뷰 기간:", analysis_overview["review_date_min"], "~", analysis_overview["review_date_max"])
print("Steam 라벨 분포:", analysis_overview["steam_label_distribution"])
print("LLM 감정 분포:", analysis_overview["llm_sentiment_distribution"])
print("High urgency 리뷰 수:", analysis_overview["high_urgency_review_count"])
print()

check_cols = [
    "issue_name_kor",
    "action_group_hint",
    "rule_priority_hint",
    "affected_review_count",
    "negative_mixed_review_count",
    "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count",
    "early_playtime_negative_mixed_review_count",
]
check_cols = [col for col in check_cols if col in selected_evidence.columns]

print("선택된 근거 데이터")
print(selected_evidence[check_cols].to_string(index=False))
print()

print("프롬프트 앞부분")
print(patch_ops_prompt[:2500])

분석 대상: Tainted Grail: The Fall of Avalon
분석 리뷰 수: 1000
분석 리뷰 기간: 2026-01-29 ~ 2026-04-29
Steam 라벨 분포: positive: 671, negative: 329
LLM 감정 분포: positive: 592, negative: 302, mixed: 103, neutral: 3
High urgency 리뷰 수: 178

선택된 근거 데이터
issue_name_kor action_group_hint rule_priority_hint  affected_review_count  negative_mixed_review_count  high_urgency_review_count  recent_30d_negative_mixed_review_count  early_playtime_negative_mixed_review_count
            버그             즉시 확인                  상                     95                           85                         38                                      33                                          11
            성능             즉시 확인                  상                     60                           53                         26                                      22                                          13
           크래시             즉시 확인                  상                     30                           29                      

# 10. LLM 패치·운영 전략 생성

In [14]:

# ============================================================
# LLM 패치·운영 전략 생성
# ============================================================
# RUN_PATCH_OPS_LLM=False이면 실제 LLM 호출은 하지 않는다.

if RUN_PATCH_OPS_LLM:
    if patch_ops_agent is None:
        raise RuntimeError(
            "patch_ops_agent가 생성되지 않았습니다. "
            ".env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
        )

    result = await patch_ops_agent.run(
        patch_ops_prompt,
        model_settings=patch_ops_settings,
    )

    patch_ops_output = result.output
    patch_ops_result_dict = to_serializable(patch_ops_output)

    print("LLM 패치·운영 전략 생성 완료")

else:
    patch_ops_output = None
    patch_ops_result_dict = None

    print("RUN_PATCH_OPS_LLM=False")
    print("LLM 호출은 하지 않고, 프롬프트와 근거 데이터만 생성했습니다.")

LLM 패치·운영 전략 생성 완료


# 11. LLM 결과 검증 및 표 생성

In [15]:

# ============================================================
# LLM 결과 검증 및 표 생성
# ============================================================
# 목적:
# LLM이 action_group 또는 priority를 잘못 옮기더라도,
# 최종 출력에서는 04-1 근거표의 action_group_hint, rule_priority_hint를 다시 적용한다.


def build_fixed_maps(evidence_df):
    work = evidence_df.copy()

    priority_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["rule_priority_hint"]
        .to_dict()
    )

    action_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["action_group_hint"]
        .to_dict()
    )

    evidence_summary_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["priority_reason"]
        .to_dict()
    )

    return priority_map, action_map, evidence_summary_map


priority_map, action_map, evidence_summary_map = build_fixed_maps(selected_evidence)


def validate_patch_ops_result(result_dict, evidence_df):
    priority_map, action_map, _ = build_fixed_maps(evidence_df)

    item_keys = [
        "immediate_actions",
        "short_term_improvements",
        "long_term_reviews",
        "strengths_to_keep",
    ]

    warnings = []

    for key in item_keys:
        for item in result_dict.get(key, []):
            issue_name = item.get("issue_name", "")
            item_priority = item.get("priority", "")
            item_action_group = item.get("action_group", "")

            fixed_priority = priority_map.get(issue_name)
            fixed_action = action_map.get(issue_name)

            if fixed_priority is None or fixed_action is None:
                warnings.append(f"근거표에 없는 이슈가 LLM 결과에 포함됨: {issue_name}")
                continue

            if item_priority != fixed_priority:
                warnings.append(
                    f"이슈 '{issue_name}'의 priority={item_priority}, 근거표 rule_priority_hint={fixed_priority}"
                )

            if item_action_group != fixed_action:
                warnings.append(
                    f"이슈 '{issue_name}'의 action_group={item_action_group}, 근거표 action_group_hint={fixed_action}"
                )

    return warnings


def make_patch_ops_strategy_table(result_dict, evidence_df, drop_unknown_issues=True):
    priority_map, action_map, evidence_summary_map = build_fixed_maps(evidence_df)

    rows = []
    item_keys = [
        "immediate_actions",
        "short_term_improvements",
        "long_term_reviews",
        "strengths_to_keep",
    ]

    for key in item_keys:
        for item in result_dict.get(key, []):
            issue_name = item.get("issue_name", "")

            if issue_name not in priority_map:
                if drop_unknown_issues:
                    continue
                fixed_priority = item.get("priority", "")
                fixed_action = item.get("action_group", "")
            else:
                fixed_priority = priority_map[issue_name]
                fixed_action = action_map[issue_name]

            detailed_actions = item.get("detailed_actions", [])
            if isinstance(detailed_actions, list):
                detailed_actions_text = "\n".join([f"- {x}" for x in detailed_actions])
            else:
                detailed_actions_text = str(detailed_actions)

            rows.append({
                "대응 구분": fixed_action,
                "우선순위": fixed_priority,
                "이슈": issue_name,
                "패치·운영 방향": item.get("patch_ops_direction", ""),
                "세부 실행안": detailed_actions_text,
                "근거 요약": item.get("evidence_summary", evidence_summary_map.get(issue_name, "")),
                "기대 효과": item.get("expected_effect", ""),
                "주의사항": item.get("caution", ""),
            })

    strategy_df = pd.DataFrame(rows)

    if len(strategy_df) == 0:
        return strategy_df

    priority_order_map = {"상": 1, "중": 2, "하": 3}
    action_order_map = {
        "즉시 확인": 1,
        "단기 개선": 2,
        "장기 검토": 3,
        "강점 유지": 4,
        "검토 필요": 5,
    }

    strategy_df["action_order"] = strategy_df["대응 구분"].map(action_order_map).fillna(9)
    strategy_df["priority_order"] = strategy_df["우선순위"].map(priority_order_map).fillna(9)
    strategy_df = strategy_df.sort_values(["action_order", "priority_order", "이슈"])
    strategy_df = strategy_df.drop(columns=["action_order", "priority_order"])

    return strategy_df


if patch_ops_result_dict is not None:
    validation_warnings = validate_patch_ops_result(
        patch_ops_result_dict,
        evidence_df=selected_evidence,
    )

    if validation_warnings:
        display(Markdown("### LLM 출력 검증 결과"))
        display(Markdown("LLM 출력이 근거표와 다르게 나온 항목이 있어, 최종 표에서는 04-1 근거표의 대응 구분과 우선순위를 적용했다."))
        for warning in validation_warnings:
            display(Markdown(f"- {warning}"))

    patch_ops_strategy_df = make_patch_ops_strategy_table(
        patch_ops_result_dict,
        evidence_df=selected_evidence,
        drop_unknown_issues=True,
    )

    print("패치·운영 전략 표 행 수:", len(patch_ops_strategy_df))
    display(patch_ops_strategy_df)
else:
    validation_warnings = []
    patch_ops_strategy_df = pd.DataFrame()
    print("LLM 결과가 없어 패치·운영 전략 표를 생성하지 않았습니다.")

패치·운영 전략 표 행 수: 14


,대응 구분,우선순위,이슈,패치·운영 방향,세부 실행안,근거 요약,기대 효과,주의사항
0,즉시 확인,상,버그,"반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.",- 반복 보고되는 퀘스트 NPC 스폰 버그 재현 및 수정\n- 주요 분기점의 트리거 로직 정밀 조사 및 소프트 락 방지 패치 배포\n- 플레이 방해 수준이 높은 크래시 이슈 우선 수정,"영향 리뷰 95개, 부정·혼합 85개, High urgency 38개, 최근 30일 부정·혼합 33개로 기술적 결함이 매우 심각함.",게임 진행 차단 현상을 해소하여 유저의 플레이 경험 안정화 및 환불률 감소.,재현 경로가 불분명한 경우 유저의 시스템 사양과 로그 파일을 우선적으로 확보하십시오.
1,즉시 확인,상,성능,"프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.",- 스팀 덱 및 저사양 환경에서의 프레임 드랍 원인 분석 및 최적화 패치\n- 오픈 월드 구간의 리소스 로딩 및 렌더링 최적화 점검\n- 마이크로 스터터링 현상 재현 테스트 및 엔진 설정 개선,"영향 리뷰 60개, 부정·혼합 53개, High urgency 26개, 최근 30일 부정·혼합 22개로 성능 이슈가 지속적으로 제기됨.",프레임 안정성 확보를 통해 전반적인 게임 플레이 쾌적도 향상.,스팀 덱 등 특정 하드웨어 환경에서의 성능 저하를 별도로 모니터링하십시오.
3,즉시 확인,상,성장/반복 노가다,반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.,- 장비 업그레이드 비용 및 아이템 드랍 테이블 재조정\n- 몬스터별 난이도 곡선 및 보상 체계 개선\n- 레벨업과 장비 성장이 유의미하게 느껴지도록 스케일링 시스템 개선,"영향 리뷰 28개, 부정·혼합 26개, High urgency 14개, 최근 30일 부정·혼합 9개로 성장 체감 부족에 대한 불만이 높음.",반복 노가다 피로도를 줄이고 탐험의 보상 만족도 증대.,성장 속도 조정 시 기존 유저의 박탈감이 발생하지 않도록 주의하십시오.
4,즉시 확인,상,최적화,최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.,- 텍스처 팝인 및 LOD 설정 최적화 패치 배포\n- 초기 실행 시 발생하는 블랙 스크린 및 응답 없음 문제 호환성 체크\n- 성능 개선 패치 로드맵 공유를 통한 유저 신뢰 회복,"영향 리뷰 27개, 부정·혼합 26개, High urgency 14개, 최근 30일 부정·혼합 11개로 최적화에 대한 유저 불만이 지속됨.",기술적 완성도 향상을 통해 사양 진입 장벽 완화 및 플레이 경험 개선.,특정 사양에서만 발생하는 이슈인지 확인하기 위해 다양한 환경에서의 테스트가 필요합니다.
2,즉시 확인,상,크래시,"크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.",- 크래시 리포트 로그 분석을 통한 메모리 누수 및 충돌 원인 파악\n- 특정 지역(Cuanacht 등) 진입 시 발생하는 크래시 핫픽스 배포\n- 자동 저장 기능의 기술적 결함 수정 및 안정성 강화,"영향 리뷰 30개, 부정·혼합 29개, High urgency 26개, 최근 30일 부정·혼합 11개로 크래시 발생 시 유저 이탈이 매우 높음.",게임 강제 종료 현상을 최소화하여 플레이 지속성 확보.,세이브 데이터 손실 방지를 위해 자동 저장 시스템의 안정성을 최우선으로 점검하십시오.
7,단기 개선,상,UI/UX,"메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다.",- 인벤토리 정렬 및 포션 스택 문제 해결을 위한 UI 개선\n- 퀘스트 마커 및 맵 마커 시스템 추가를 통한 편의성 강화\n- 메뉴 구성 간소화 및 직관적인 조작 안내 제공,"영향 리뷰 63개, 부정·혼합 58개, High urgency 20개, 최근 30일 부정·혼합 19개로 편의성 개선 요구가 지속됨.",유저들의 게임 접근성 향상 및 불필요한 피로도 감소.,UI/UX 개선은 유저 편의성을 높이는 것이 목적이므로 직관성을 최우선으로 고려하십시오.
5,단기 개선,상,게임플레이 루프,"반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.",- 전투 조작감 및 타격 판정 로직 점검 및 개선\n- 소환수 AI 및 밸런스 조정(마나 소모량 등)을 통한 전투 긴장감 확보\n- 반복적인 플레이 루프를 개선하기 위한 목표 구조 및 보상 흐름 재설계,"영향 리뷰 284개, 부정·혼합 208개, High urgency 73개, 최근 30일 부정·혼합 81개로 게임플레이 루프에 대한 개선 요구가 가장 많음.",전투의 재미와 몰입감을 높여 유저들의 플레이 지속 시간 증대.,전투 시스템 변경은 밸런스에 큰 영향을 미치므로 충분한 테스트가 선행되어야 합니다.
8,단기 개선,상,난이도,초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.,- 초반 진입 장벽 완화를 위한 적 레벨 스케일링 재검토\n- 후반부 난이도 피로도를 낮추기 위한 옵션 제공 및 안내 보강\n- 불합리한 보스전 판정 및 난이도 밸런스 재조정,"영향 리뷰 68개, 부정·혼합 48개, High urgency 19개, 최근 30일 부정·혼합 21개로 난이도에 대한 불만이 존재함.",신규 유저의 진입 장벽 완화 및 후반부 플레이 만족도 향상.,난이도 옵션 추가 시 게임의 핵심 재미가 훼손되지 않도록 주의하십시오.
6,단기 개선,상,밸런스,"전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다.","- 전투, 성장, 보상, 적 난이도의 불균형 지점 조정\n- 보스와 일반 몬스터 간의 위협 수준 재검토\n- 적의 체력 수치만 높이는 방식의 난이도 조절 지양 및 스케일링 개선","영향 리뷰 130개, 부정·혼합 118개, High urgency 40개, 최근 30일 부정·혼합 50개로 밸런스 불균형이 심각함.",전투의 공정성 확보 및 유저들의 난이도 만족도 향상.,난이도 조정 시 초보자와 숙련자 모두를 고려한 옵션 제공을 검토하십시오.
9,장기 검토,상,그래픽/사운드,그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.,- 오픈 월드 환경의 텍스처 스트리밍 및 렌더링 품질 개선 검토\n- 근거리 팝인 현상 및 셰이더 떨림 현상 엔진 차원 점검\n- 그래픽 품질 대비 리소스 점유율 최적화 로드맵 수립,"영향 리뷰 133개, 부정·혼합 95개, High urgency 28개, 최근 30일 부정·혼합 38개로 그래픽 품질에 대한 개선 요구가 많음.",시각적 몰입감 향상 및 게임의 전반적인 완성도 제고.,"그래픽 개선은 리소스 소모가 크므로, 몰입을 방해하는 핵심 요소부터 단계적으로 접근하십시오."


# 12. 보고서용 출력 및 저장

In [16]:

# ============================================================
# 보고서용 Markdown 생성 함수
# ============================================================

def build_report_markdown(result_dict, strategy_df):
    lines = []

    lines.append(f"# {result_dict.get('title', '출시 후 패치·운영 전략 제안')}")
    lines.append("")

    for title, key in [
        ("게임 요약", "game_summary"),
        ("데이터 요약", "data_summary"),
        ("현재 반응 상태", "current_status_summary"),
    ]:
        value = result_dict.get(key, "")
        if value:
            lines.append(f"## {title}")
            lines.append(value)
            lines.append("")

    if len(strategy_df) > 0:
        lines.append("## 패치·운영 전략 표")
        try:
            strategy_table_text = strategy_df.to_markdown(index=False)
        except Exception:
            strategy_table_text = strategy_df.to_csv(index=False)
        lines.append(strategy_table_text)
        lines.append("")

    operation_notes = result_dict.get("operation_notes", [])
    if operation_notes:
        lines.append("## 운영 관점 제안")
        for note in operation_notes:
            lines.append(f"- {note}")
        lines.append("")

    cautions = result_dict.get("cautions", [])
    if cautions:
        lines.append("## 해석 시 주의사항")
        for caution in cautions:
            lines.append(f"- {caution}")
        lines.append("")

    final_summary = result_dict.get("final_summary", "")
    if final_summary:
        lines.append("## 최종 요약")
        lines.append(final_summary)
        lines.append("")

    return "\n".join(lines)
